# M2 실제148 Pose: T4 DDIM5/10 재실행

기존 결과는 이미 실행 완료했습니다. 이 notebook은 선택적 재현용이며 학습 셀이 없습니다. T4 runtime을 선택하고 위에서 아래로 실행합니다. Drive 연결은 필요 없습니다. 제공된 code ZIP과 verified input ZIP만 업로드합니다. 새 notebook 자체는 문법 검증만 완료했으며 실제 실행 근거는 별도 executed notebook입니다.

In [ ]:
# 1. T4 환경 확인
import torch, numpy, matplotlib, PIL
assert torch.cuda.is_available(), 'CUDA unavailable: stop'
assert 'T4' in torch.cuda.get_device_name(0), 'Select T4 for comparable timing'
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
# 2. 코드 ZIP 및 검증 입력 ZIP 업로드
from google.colab import files
uploaded = files.upload()
# REWIND_M2_real148_code.zip, M2_verified_148_input.zip 두 파일 선택


In [ ]:
# 3. SHA 검사 및 새 작업 폴더 준비
import hashlib, json, subprocess, sys, uuid, zipfile
from pathlib import Path
CODE = Path('/content/REWIND_M2_real148_code.zip')
INPUT = Path('/content/M2_verified_148_input.zip')
assert hashlib.sha256(CODE.read_bytes()).hexdigest() == '372c00d91eba81678f8db74d69e1ffff24a7c746819f37a2f3adbc8eeeff9a4a'
assert hashlib.sha256(INPUT.read_bytes()).hexdigest() == '184967a9732702fc0459350974b87e67f6499144dcfdbec48b18cb657805d74d'
BASE = Path('/content') / ('m2_real_' + uuid.uuid4().hex)
ROOT = BASE / 'repo'
ROOT.mkdir(parents=True)
with zipfile.ZipFile(CODE) as z:
    assert z.testzip() is None
    assert sum(i.file_size for i in z.infolist()) < 16*1024**2
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(ROOT)
RESULT = BASE / 't4_result'
print('INPUT_SHA_PASS', 'CODE_SHA_PASS', str(RESULT))


In [ ]:
# 4. DDIM5/10 추론 및 Baseline 재현 검사
subprocess.run([sys.executable,str(ROOT/'tools/final_m2_real.py'),'--input-bundle',str(INPUT),'--output',str(RESULT),'--device','cuda'],cwd=ROOT,check=True)
evaluation=json.loads((RESULT/'evaluation.json').read_text())
assert evaluation['status']=='completed'
assert all(v['status']=='PASS' for v in evaluation['baseline_reproduction'].values())
print(json.dumps(evaluation['results'],indent=2))


In [ ]:
# 5. 실제 출력 그래프와 Motion GIF
subprocess.run([sys.executable,str(ROOT/'tools/final_m2_plot.py'),str(RESULT),'--gifs'],cwd=ROOT,check=True)
from IPython.display import Image, display
display(Image(filename=str(RESULT/'figures/joint_error.png')))


In [ ]:
# 6. 결과 ZIP 저장 및 다운로드
RESULT_ZIP=BASE/'REWIND_M2_T4_real148_rerun_results.zip'
with zipfile.ZipFile(RESULT_ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RESULT.rglob('*')):
        if p.is_file() and 'private_input' not in p.relative_to(RESULT).parts:
            z.write(p,p.relative_to(RESULT))
print('RESULT_ZIP', str(RESULT_ZIP))
print('SHA256',hashlib.sha256(RESULT_ZIP.read_bytes()).hexdigest())
files.download(str(RESULT_ZIP))


## 정상 출력과 중단 조건

정상: INPUT_SHA_PASS/CODE_SHA_PASS, status=completed, train/holdout Baseline PASS, 네 행의 지표와 결과 ZIP. 이번 실측 Opening은5-step 12.228817raw,10-step 12.232006raw이며 지연은 환경에 따라 달라집니다. Hash 불일치, T4 부재, shape/manifest 실패, 재현 FAIL, 시간 예산 초과 시 중단하고 오류를 보존하세요. Guard를 수정하지 마세요. 수정할 데이터 설정값은 없고 Drive 경로는 사용하지 않습니다. 결과 ZIP은 셀6의 RESULT_ZIP 절대 경로로 출력됩니다.